# Fabric Notebook Scanner v2

_Thin orchestrator over `fabric-scanner==0.3.3`._

Five cells:
1. **Install** the scanner wheel.
2. **Configure** the scan (the only cell most users touch).
3. **Probe** the source — sanity-check paths or endpoints.
4. **Run** the scan — produces the inventory and findings Delta tables.
5. **Explore** the findings — display the top rollups.

All scanner logic lives in the installable wheel; edits to detection
patterns or schema happen there, not in this notebook.

In [ ]:
# Install the scanner wheel (re-run when you bump the version).
%pip install -q fabric-scanner==0.3.3

## 1. Configuration

`ScannerConfig` is a frozen dataclass — every knob has a sensible
default. The most common edits are commented inline below.

In [ ]:
from fabric_scanner import ScannerConfig

cfg = ScannerConfig(
    # --- Where to read notebooks from ---
    source_mode      = "lakehouse",   # or "api" for tenant scan
    source_layout    = "flat",        # or "ws_dated"
    source_subpath   = "Files/notebooks",
    source_file_glob = "*.ipynb",
    # Leave source_lakehouse_* blank to auto-detect the mounted
    # default lakehouse (recommended). Set explicitly to scan a
    # different LH the notebook isn't attached to.
    # source_lakehouse_workspace_id = "<ws-guid>",
    # source_lakehouse_id           = "<lh-guid>",

    # --- Where to write outputs ---
    write_to_default_lakehouse = True,   # saveAsTable on attached LH
    # write_workspace_id = "<ws-guid>",
    # write_lakehouse_id = "<lh-guid>",
    inventory_table = "v2_inventory",
    output_table    = "v2_findings",

    # --- Behavior ---
    min_severity              = "low",
    scan_markdown_and_outputs = True,
    target_partition_size     = 200,
)
cfg

## 2. Diagnostics — probe paths / endpoints

Prints a one-screen summary of what the scanner *will* read.
Safe to run before kicking off the actual scan.

In [ ]:
from fabric_scanner import resolve_paths, probe

resolved = resolve_paths(cfg)
probe(cfg, resolved)

## 3. Run the scan

Builds the inventory, fans out the partition scanners, writes
the inventory + findings Delta tables, and returns a
`ScannerResult` with summary stats and the 10 rollup DataFrames.

In [ ]:
from fabric_scanner.spark import run

result = run(cfg, spark)

print(f"Notebooks scanned : {result.notebook_count:,}")
print(f"Findings rows     : {result.findings_count:,}")
print(f"Inventory table   : {result.inventory_table}")
print(f"Findings table    : {result.findings_table}")
print()
print("By severity:")
for sev, n in sorted(result.by_severity.items()):
    print(f"  {sev:10s}  {n:>6,}")

## 4. Explore the findings

Each rollup is a regular Spark DataFrame — display, filter, or
join as you like. See `fabric_scanner.persist.SummaryReport`
for the full list.

In [ ]:
display(result.summary.by_severity)

In [ ]:
display(result.summary.review_queue)

In [ ]:
display(result.summary.cross_workspace_flows)

In [ ]:
display(result.summary.sample_critical)